# 1.5 — Ingesta: Enfermedades respiratorias (variable de respuesta)

Casos confirmados de IRA — infecciones
respiratorias agudas (CIE-10 J00–J06, J20, J21, menos J02.0 y J03.0) —
de la Secretaría de Salud de Sonora, formato SUIVE, 2010–2020. Es
**archivo manual** (no se descarga): `data/raw/salud/enf_resp_sonora_2010_2020.csv`.

Asma, EPOC y neumonías **no están disponibles** en la fuente abierta; el
análisis se hace solo con IRA.

## Entradas
- `data/raw/salud/enf_resp_sonora_2010_2020.csv` — semana epidemiológica
  × municipio × sexo × grupo de edad × CLUES (unidad médica que reporta).
- `data/external/clues/clues_hermosillo_ira.csv` — catálogo CLUES para
  georreferenciar las unidades (se genera abajo).

## Salidas
- `data/interim/salud/` — tabla depurada que usa el notebook `2.5`.


In [3]:
from pathlib import Path
import sys
import os 
import pandas as pd

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Definir rutas y parámetros
from sonora_health_climate_eda.config import (
    RAW_DATA_DIR,
    INTERIM_DATA_DIR
)

main_folder = os.path.join(INTERIM_DATA_DIR, "salud")
os.makedirs(main_folder, exist_ok=True)

from sonora_health_climate_eda.schemas import InfeccionesRespiratoriasSonoraSchema # improtar el esquema de validación de datos de sequía

# Casos confirmados de IRA (CIE-10 J00-J06/J20/J21 menos J02.0/J03.0)
# semanal epidemiologica x municipio -> mensual para Hermosillo.
# Fuente local: data/raw/salud/enf_resp_sonora_2010_2020.csv
# (Secretaria de Salud de Sonora, cobertura 2010-2020; ver
#  references/salud/data_dictionary_enf_resp_sonora.md).
# La ventana mensual queda definida por PANEL_START_YEAR/PANEL_END_YEAR.

enfermedades_respiratorias_sonora = pd.read_csv(os.path.join(RAW_DATA_DIR, "salud", "enf_resp_sonora_2010_2020.csv"))



## Catálogo CLUES (DGIS)

Las unidades que reportan casos se identifican con su **CLUES** (Clave Única
de Establecimientos de Salud). El comando `download-clues-data` descarga el
catálogo maestro DGIS (~26 MB) a `data/raw/clues/` y genera el extracto
`data/external/clues/clues_hermosillo_ira.csv` que usa el notebook 2.5 para
georreferenciar las unidades. Si el CSV de IRA ya está disponible, el extracto
contiene solo las unidades que reportaron IRA en Hermosillo; si no, todos los
establecimientos del municipio.

**Fuente: SS/DGIS, catálogo ESTABLECIMIENTO_SALUD, actualización 2026-10.**

In [ ]:
from sonora_health_climate_eda.dataset import download_clues_data

download_clues_data()

## Validación de la lectura

Cada fila del CSV se valida con `InfeccionesRespiratoriasSonoraSchema`
(de `schemas.py`): año y semana en rango, municipio presente y casos no
negativos.

### Función de validación

Recorre la tabla fila por fila: las que cumplen el schema pasan y las que
no se reportan con su error.

In [4]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            df_infecciones_respiratorias = InfeccionesRespiratoriasSonoraSchema(**row.to_dict())
            validated_rows.append(df_infecciones_respiratorias.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [5]:
df_validated = validate_df(enfermedades_respiratorias_sonora)
print(f"Filas válidas: {len(df_validated):,} de {len(enfermedades_respiratorias_sonora):,}")
df_validated.head(5)

Filas válidas: 1,732,444 de 1,732,444


,clave_municipio,municipio,clave_entidad,anio,semana,genero,grupo_edad,clave_enfermedad,clues,casos
0,1,ACONCHI,26,2010,1,Masculino,De 00 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMB000021,1
1,1,ACONCHI,26,2010,1,Masculino,De 05 a 09 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMS000010,1
2,1,ACONCHI,26,2010,1,Femenino,De 15 a 19 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMB000021,1
3,1,ACONCHI,26,2010,1,Masculino,De 50 a 59 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMS000010,1
4,1,ACONCHI,26,2010,1,Femenino,De 50 a 59 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMS000010,1
